# Близкие фразы: проверяем косинус

Дополнение к § 3.3. Сначала разберём косинус на учебных векторах, затем запустим классификатор тональности и многоязычную модель эмбеддингов на коротких русских и белорусских фразах. Классификация возвращает метку; поиск по эмбеддингам ранжирует кандидатов по близости.


## Запуск полного опыта

Установите пакеты из `requirements.txt` со страницы главы. При первом запуске ноутбук скачает веса классификатора и энкодера из интернета; модельные ячейки выполняются на CPU. В ячейке «Опыт с русским и белорусским текстом» установите `run_model = True`, затем выполните ячейки сверху вниз. Сначала запишите прогнозы, после запуска сопоставьте их с выводом. Числовые упражнения выше работают отдельно от загружаемых моделей.


## Сходство и направление

Посчитайте косинус между запросом и тремя учебными векторами. Объясните, почему противоположный вектор получает отрицательное значение.


In [ ]:
import numpy as np
query = np.array([1.0,0.0])
candidates = np.array([[0.8,0.6],[0,1],[-1,0]], dtype=float)
def cosine(a,b):
    return float(np.dot(a,b) / (np.linalg.norm(a)*np.linalg.norm(b)))
scores = [cosine(query,row) for row in candidates]
print(np.round(scores,3))
assert np.allclose(scores, [0.8,0,-1])

## Изменяем запрос

Поменяйте запрос на $(0,1)$ и пересчитайте порядок кандидатов. Косинус показывает относительную близость векторов, а не вероятность совпадения смыслов.


In [ ]:
new_query = np.array([0.0,1.0])
new_scores = [cosine(new_query,row) for row in candidates]
print(sorted(enumerate(new_scores), key=lambda x: (-x[1],x[0])))

## Подготовим сравнение фраз

До запуска предскажите сходство для перефразирования и для фразы с отрицанием. Затем посмотрите на их смысл и не делайте вывод только по числу.


In [ ]:
phrases = [("Когда забрать фигурку?", "Калі забраць фігурку?"),
           ("Когда забрать фигурку?", "Не нужно забирать фигурку")]
for left,right in phrases:
    print(left, "→", right)

## Опыт с русским и белорусским текстом

Сначала запишите ожидаемые метки четырёх русских отзывов и порядок сходства для пар фраз. Включите `run_model` и запустите модельные ячейки: классификатор покажет метки и оценки, энкодер — матрицу косинусных сходств. Классификатор здесь используется для русских отзывов, а многоязычный энкодер отдельно сравнивает русский и белорусский текст.


In [ ]:
run_model = False
if run_model:
    import pandas as pd
    from transformers import pipeline
    from sentence_transformers import SentenceTransformer

    model_id = "blanchefort/rubert-base-cased-sentiment"
    revision = "2d4b5fc431489a87ca4e44b842d904058706e8d4"
    reviews = ["Фигурка получилась красивой", "Фигурка не получилась красивой",
               "Фигурку напечатали вчера", "Отлично, снова сломался принтер"]
    human = ["POSITIVE", "NEGATIVE", "NEUTRAL", None]
    classifier = pipeline("text-classification", model=model_id, revision=revision, device=-1)
    answers = classifier(reviews)
    report = pd.DataFrame({"text": reviews, "human": human,
                           "model": [a["label"] for a in answers],
                           "score": [a["score"] for a in answers]})
    print(report.to_string(index=False))
    clear = report[report["human"].notna()]
    print("Совпадение с разметкой:", (clear["human"] == clear["model"]).mean())
    encoder_id = "sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2"
    encoder_revision = "e8f8c211226b894fcb81acc59f3b34ba3efd5f42"
    encoder = SentenceTransformer(encoder_id, revision=encoder_revision, device="cpu")
    phrases = ["Фигурку нужно забрать сегодня", "Сегодня нужно получить фигурку",
               "Фигурку не нужно забирать сегодня", "Сегодня идёт математический кружок",
               "Калі забраць фігурку?", "Когда забрать фигурку?"]
    vectors = encoder.encode(phrases, normalize_embeddings=True)
    similarity = vectors @ vectors.T
    print(pd.DataFrame(similarity, index=range(len(phrases)),
                       columns=range(len(phrases))).round(3).to_string())
    for i, phrase in enumerate(phrases): print(i, phrase)


## Разбор результата

Сопоставьте три однозначные человеческие метки с ответами классификатора; ироничный отзыв оставьте без метки, пока не согласовано правило. В матрице сравните перефразирование, отрицание, другую тему и русско-белорусскую пару. Объясните, для какой задачи высокая близость полезна, а в каком случае она может привести к неверному ответу.


In [ ]:
if run_model:
    print("Перефразирование:", round(float(similarity[0, 1]), 3))
    print("Отрицание:", round(float(similarity[0, 2]), 3))
    print("Другая тема:", round(float(similarity[0, 3]), 3))
    print("Русский и белорусский вопросы:", round(float(similarity[4, 5]), 3))


In [ ]:
if run_model:
    from importlib.metadata import version
    print("Библиотеки:", {name: version(name) for name in
          ["torch", "torchvision", "transformers", "numpy", "pandas"]})
    print("Ревизии:", model_id, revision, encoder_id, encoder_revision)


## Самопроверка




### 1. Классификатор вернул метку NEGATIVE для отзыва. Что именно сообщает этот результат?

A. Что модель отнесла отзыв к отрицательному классу.

B. Что событие в отзыве действительно произошло.

C. Что отзыв похож на все отрицательные тексты.

D. Что модель нашла похожую фразу.




### 2. Два читателя по-разному поняли ироничный отзыв. Как поступить перед оценкой модели?

A. Взять ответ модели за правильную метку.

B. Согласовать правило разметки и отметить неоднозначность.

C. Удалить все отзывы с отрицанием.

D. Считать оба ответа правильными без правила.




### 3. Векторы $a=(1,1)$ и $b=(1,0)$ сравнивают по косинусу. Какое значение получится примерно?

A. 0,500

B. 0,707

C. 1,000

D. 1,414




### 4. Оба эмбеддинга нормированы до длины 1. Что тогда даёт их скалярное произведение?

A. Косинусное сходство.

B. Вероятность одинакового смысла.

C. Расстояние между текстами в словах.

D. Метка тональности.




### 5. Кандидаты получили сходства 0,81; 0,76 и 0,42. Как интерпретировать первый результат?

A. Первый кандидат гарантированно является синонимом запроса.

B. Первый кандидат ближе остальных в этой модели и среди этих кандидатов.

C. Вероятность правильного ответа равна 81 %.

D. Первый кандидат обязательно имеет ту же тональность.




### 6. Для запроса «Фигурку нужно забрать сегодня» фраза с «не нужно» получила высокий косинус. Система должна находить взаимозаменяемые ответы. Как оценить результат?

A. Это удачный ответ: слова почти одинаковы.

B. Это ошибка для такой цели: отрицание меняет действие.

C. Это подтверждает, что косинус — вероятность.

D. Нужно удалить из запроса все глаголы.




### 7. Многоязычный энкодер сблизил русскую и белорусскую версии вопроса. Какой вывод оправдан?

A. Модель одинаково качественна на всех языках.

B. Этот пример показывает близость двух фраз; качество на языке нужно проверять на наборе примеров.

C. Русский классификатор тоже готов к белорусским отзывам.

D. Перевод больше не нужен ни в каких задачах.




### 8. На трёх однозначно размеченных отзывах модель угадала два. Какова доля верных ответов на этой мини-выборке?

A. 1/3

B. 2/3

C. 2/2

D. 3/2




### 9. Вы хотите сравнить два энкодера на одинаковых запросах. Что следует сохранить одинаковым?

A. Кандидатов, запросы и правило оценки.

B. Только названия моделей.

C. Для каждой модели выбрать свой набор кандидатов.

D. Менять запрос до получения желаемого порядка.




### 10. Русскоязычный классификатор применили к белорусскому сообщению и получили уверенную метку. Какой следующий шаг разумен?

A. Принять метку: высокая оценка заменяет проверку.

B. Проверить модель на белорусских примерах с известными метками.

C. Использовать её ответ как эталон для всех примеров.

D. Перейти к поиску эмбеддингов без проверки.




## Подсказки и решения

### Вопрос 1

**Подсказка**


Сопоставьте метку с набором классов задачи.



**Решение**


**A.** Классификатор присваивает одну метку из заданного набора; он не устанавливает истинность сообщения.



### Вопрос 2

**Подсказка**


Сравнивать модель можно лишь с определённой эталонной меткой.



**Решение**


**B.** Сначала нужно определить критерий разметки; иначе доля верных ответов зависит от произвольного выбора.



### Вопрос 3

**Подсказка**


Скалярное произведение равно 1, а длина $a$ равна $\sqrt{2}$.



**Решение**


**B.** $\operatorname{cos}(a,b)=1/(\sqrt{2}\cdot1)\approx0{,}707$.



### Вопрос 4

**Подсказка**


В формуле косинуса знаменатель — произведение длин векторов.



**Решение**


**A.** При единичных длинах знаменатель равен 1, поэтому скалярное произведение равно косинусу.



### Вопрос 5

**Подсказка**


Интерпретируйте значение только как ранжирование по выбранной мере.



**Решение**


**B.** Косинус задаёт близость в пространстве модели, но не доказывает совпадение смысла.



### Вопрос 6

**Подсказка**


Спросите, сохраняется ли смысл, а не только похожи ли слова.



**Решение**


**B.** Для поиска взаимозаменяемых ответов отрицание принципиально; высокая близость здесь опасна.



### Вопрос 7

**Подсказка**


Один пример не заменяет оценку на размеченном наборе.



**Решение**


**B.** Результат относится к конкретным фразам и модели, а не доказывает общее качество.



### Вопрос 8

**Подсказка**


Разделите число верных ответов на общее число примеров.



**Решение**


**B.** Доля равна $2/3$. Это описывает только три проверенных примера.



### Вопрос 9

**Подсказка**


Для честного сравнения меняйте модель, сохраняя данные и критерий.



**Решение**


**A.** Тогда различия в ранжировании можно связать с энкодером.



### Вопрос 10

**Подсказка**


Проверьте работу на языке, где планируете применять модель.



**Решение**


**B.** Оценку нужно провести на размеченных белорусских данных; уверенность модели сама по себе качество не подтверждает.



